# Study Abroad Finder
### Google ADK + 4 MCP Servers

---

| MCP Server | Type | Purpose |
|---|---|---|
| **Tavily MCP** | Remote (Tavily's server) | Search top universities from the web |
| **Google Maps MCP** | Remote (Google's server) | Weather + nearby places around campus |
| **Scholarship MCP** | Local (**our own server**) | Return scholarship data from our Python dict |
| **Filesystem MCP** | Local (Anthropic's server) | Save the final report to disk |

**Key Learning:** The agent connects to ALL 4 MCP servers and decides which tool to call automatically.

## Cell 1 — Install Dependencies

In [ ]:
%pip install google-adk python-dotenv mcp

## Cell 2 — Load API Keys

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

TAVILY_API_KEY      = os.getenv("TAVILY_API_KEY")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
GOOGLE_API_KEY      = os.getenv("GOOGLE_API_KEY")

print("TAVILY_API_KEY      :", "SET" if TAVILY_API_KEY      else "MISSING")
print("GOOGLE_MAPS_API_KEY :", "SET" if GOOGLE_MAPS_API_KEY else "MISSING")
print("GOOGLE_API_KEY      :", "SET" if GOOGLE_API_KEY      else "MISSING")

## Cell 3 — Connect to Tavily Remote MCP
**Concept:** Tavily runs on Tavily's servers. We connect via URL. Nothing runs on our PC.

In [ ]:
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams, StdioConnectionParams
from mcp import StdioServerParameters

tavily_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"
    )
)
tavily_tools = await tavily_toolset.get_tools()
print("Tavily MCP tools:")
for t in tavily_tools:
    print(f"  - {t.name}")

## Cell 4 — Connect to Google Maps Remote MCP
**Concept:** Google Maps MCP runs on Google's servers. We connect with an API key header.

> **Note:** If this server is slow or unavailable, the agent gracefully skips weather/places and still works with the other 3 MCP servers.

In [ ]:
try:
    maps_toolset = McpToolset(
        connection_params=StreamableHTTPConnectionParams(
            url="https://mapstools.googleapis.com/mcp",
            headers={
                "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
                "Content-Type": "application/json",
                "Accept": "application/json, text/event-stream"
            }
        )
    )
    maps_tools = await maps_toolset.get_tools()
    print(f"Google Maps MCP tools ({len(maps_tools)}):")
    for t in maps_tools:
        print(f"  - {t.name}")
except Exception as e:
    maps_toolset = None
    maps_tools = []
    print(f"WARNING: Google Maps MCP unavailable — {type(e).__name__}: {e}")
    print("Agent will skip weather/places. All other tools still work.")

## Cell 5 — Connect to Our Custom Scholarship MCP Server
**Concept:** This is the MCP server WE built. It runs locally as a Python process.
The agent connects to it via stdio — same protocol as Tavily and Google Maps.

**Our server exposes 3 tools:**
| Tool | What it does |
|------|-------------|
| `get_scholarships` | Returns scholarship funding for a country + course |
| `get_cost_of_living` | Returns monthly rent, food, transport estimates for a city |
| `get_visa_requirements` | Returns visa type, processing time, required documents for a country |

In [ ]:
import os

SCHOLARSHIP_SERVER = os.path.abspath("../scholarship_mcp_server/scholarship_server.py")

scholarship_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="python",
            args=[SCHOLARSHIP_SERVER]
        ),
        timeout=30
    )
)
scholarship_tools = await scholarship_toolset.get_tools()
print("Our Custom Scholarship MCP tools:")
for t in scholarship_tools:
    print(f"  - {t.name}: {t.description}")

## Cell 6 — Connect to Filesystem MCP
**Concept:** Runs locally via npx. Reads and writes files on our PC.

In [ ]:
REPORTS_DIR = os.path.abspath("../reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

filesystem_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", REPORTS_DIR]
        ),
        timeout=30
    )
)
filesystem_tools = await filesystem_toolset.get_tools()
print("Filesystem MCP tools:")
for t in filesystem_tools:
    print(f"  - {t.name}")

## Cell 7 — Create Agent with All 4 MCP Servers
**Concept:** One agent connects to ALL 4 MCP servers. It decides which tool to call automatically.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

maps_available = len(maps_tools) > 0

AGENT_INSTRUCTION = f"""
You are a Study Abroad Advisor.
When asked about universities:
1. Use Tavily to search top 3 universities for the course
2. Use get_scholarships to find scholarships for each country and course
3. Use get_cost_of_living to get monthly living costs for each university city
4. Use get_visa_requirements to get visa info for each university country
{"5. Use Google Maps to get weather and nearby places for each university city" if maps_available else "5. (Google Maps unavailable — skip weather step)"}
6. Save a full markdown report to: {REPORTS_DIR}
7. Give a clean summary to the student
"""

all_tools = tavily_tools + maps_tools + scholarship_tools + filesystem_tools

agent = LlmAgent(
    name="study_abroad_advisor",
    model="gemini-flash-latest",
    instruction=AGENT_INSTRUCTION,
    tools=all_tools
)

total = len(all_tools)
print(f"Agent ready with {total} tools from {'4' if maps_available else '3'} MCP servers")
print(f"  Tavily tools      : {len(tavily_tools)}")
print(f"  Maps tools        : {len(maps_tools)}  {'(active)' if maps_available else '(unavailable — skipped)'}")
print(f"  Scholarship tools : {len(scholarship_tools)}  (get_scholarships, get_cost_of_living, get_visa_requirements)")
print(f"  Filesystem tools  : {len(filesystem_tools)}")

## Cell 8 — Ask a Question
Change `QUESTION` to anything you want to research.

In [ ]:
session_service = InMemorySessionService()
await session_service.create_session(
    app_name="study_abroad_finder",
    user_id="student",
    session_id="session_1"
)

runner = Runner(
    agent=agent,
    app_name="study_abroad_finder",
    session_service=session_service
)

QUESTION = "Top universities for Machine Learning in USA"

message = types.Content(role="user", parts=[types.Part(text=QUESTION)])

print(f"Question: {QUESTION}")
print("-" * 60)

async for event in runner.run_async(
    user_id="student",
    session_id="session_1",
    new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        print(event.content.parts[0].text)

## Cell 9 — Cleanup

In [ ]:
await tavily_toolset.close()
if maps_toolset:
    await maps_toolset.close()
await scholarship_toolset.close()
await filesystem_toolset.close()

print("All MCP connections closed.")
print(f"Check your report at: {REPORTS_DIR}")

---
## What You Learned

| Cell | MCP Concept |
|------|-------------|
| Cell 3 | Connect to a **remote** MCP server via URL (Tavily) |
| Cell 4 | Connect to another **remote** MCP server with API key headers (Google Maps) |
| Cell 5 | Connect to **your own** MCP server via stdio — exposing 3 custom tools |
| Cell 6 | Connect to a **local** MCP server via npx (Filesystem) |
| Cell 7 | One agent uses tools from **all 4 MCP servers** |
| Cell 8 | Agent **decides which tool to call** — MCP is just a protocol |

**The big insight:** Your custom scholarship server looks identical to Tavily and Google Maps from the agent's perspective. MCP is a standard protocol — it does not matter who built the server.

**Our custom MCP server tools:**
- `get_scholarships(country, course)` — funding data from a Python dict
- `get_cost_of_living(city)` — monthly budget breakdown for any university city
- `get_visa_requirements(country)` — visa type, documents, work permit rules